# Model tuning and final validation

Run after notebook 02. Tune on training data, select using validation ROC-AUC, then evaluate the chosen model once on the held-out test set.

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "raw_data").is_dir() and (p / "README.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

OUTPUT_DIR = PROJECT_ROOT / "processed_data"
MODEL_DIR = OUTPUT_DIR / "models"
SPLIT_PATH = OUTPUT_DIR / "model_data_with_split.csv"
if not SPLIT_PATH.is_file():
    raise FileNotFoundError(f"Run notebook 02 first; split data not found: {SPLIT_PATH}")

RANDOM_STATE = 42
NUMERIC_FEATURES = ["TermInMonths", "LogGrossApproval", "LogJobsSupported"]
CATEGORICAL_FEATURES = ["BusinessAge", "Program", "CollateralInd", "NaicsSector", "FranchiseInd"]
data = pd.read_csv(SPLIT_PATH)
if "RevolverStatus" in data:
    CATEGORICAL_FEATURES.append("RevolverStatus")
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
required = set(FEATURES + ["Default", "Split"])
missing = required.difference(data.columns)
if missing:
    raise ValueError(f"Model split is missing required fields: {sorted(missing)}")

train = data.loc[data["Split"].eq("train")]
validation = data.loc[data["Split"].eq("validation")]
test = data.loc[data["Split"].eq("test")]
if min(len(train), len(validation), len(test)) == 0:
    raise ValueError("Train, validation, and test partitions must all contain rows.")
baseline_path = MODEL_DIR / "baseline_logistic_regression.joblib"
if not baseline_path.is_file():
    raise FileNotFoundError(f"Run notebook 02 first; baseline model not found: {baseline_path}")
X_train, y_train = train[FEATURES], train["Default"].astype(int)
X_validation, y_validation = validation[FEATURES], validation["Default"].astype(int)
X_test, y_test = test[FEATURES], test["Default"].astype(int)

def make_preprocessor():
    from sklearn.pipeline import Pipeline as SklearnPipeline
    return ColumnTransformer([
        ("numeric", SklearnPipeline([("imputer", SimpleImputer(strategy="median")),
                                     ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("categorical", SklearnPipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                         ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
         CATEGORICAL_FEATURES),
    ])

def metrics_at_threshold(model, X, y, threshold=0.5):
    probability = model.predict_proba(X)[:, 1]
    prediction = probability >= threshold
    return {"accuracy": accuracy_score(y, prediction),
            "precision": precision_score(y, prediction, zero_division=0),
            "recall": recall_score(y, prediction, zero_division=0),
            "f1": f1_score(y, prediction, zero_division=0),
            "roc_auc": roc_auc_score(y, probability)}

print(f"Train={len(train):,}, validation={len(validation):,}, test={len(test):,}")

In [ ]:
class_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
model_spaces = {
    "random_forest": (
        RandomForestClassifier(class_weight="balanced_subsample", n_jobs=-1,
                               random_state=RANDOM_STATE),
        {"model__n_estimators": [250, 400],
         "model__max_depth": [None, 12, 20],
         "model__min_samples_leaf": [2, 5, 10],
         "model__max_features": ["sqrt", 0.8]},
    ),
    "xgboost": (
        XGBClassifier(scale_pos_weight=class_ratio, eval_metric="logloss",
                      n_jobs=-1, random_state=RANDOM_STATE),
        {"model__n_estimators": [200, 350, 500],
         "model__max_depth": [3, 5, 7],
         "model__learning_rate": [0.03, 0.05, 0.1],
         "model__subsample": [0.7, 0.85, 1.0],
         "model__colsample_bytree": [0.7, 0.85, 1.0]},
    ),
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
searches = {}
validation_results = []
for name, (estimator, parameters) in model_spaces.items():
    pipeline = Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])
    search = RandomizedSearchCV(
        pipeline, parameters, n_iter=8, scoring="roc_auc", cv=cv,
        n_jobs=1, random_state=RANDOM_STATE, refit=True, verbose=1,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    validation_results.append({"model": name, "cv_best_roc_auc": search.best_score_,
                               **metrics_at_threshold(search.best_estimator_, X_validation, y_validation)})
    joblib.dump(search.best_estimator_, MODEL_DIR / f"tuned_{name}.joblib")

baseline = joblib.load(baseline_path)
validation_results.append({"model": "logistic_regression_baseline", "cv_best_roc_auc": np.nan,
                           **metrics_at_threshold(baseline, X_validation, y_validation)})

validation_metrics = pd.DataFrame(validation_results).sort_values("roc_auc", ascending=False)
validation_metrics.to_csv(OUTPUT_DIR / "tuned_validation_metrics.csv", index=False)
selected_name = validation_metrics.iloc[0]["model"]
selected_model = baseline if selected_name == "logistic_regression_baseline" else searches[selected_name].best_estimator_
print(f"Selected on validation ROC-AUC: {selected_name}")
display(validation_metrics)

In [ ]:
test_results = metrics_at_threshold(selected_model, X_test, y_test, threshold=0.5)
test_results = pd.DataFrame([{"model": selected_name, "threshold": 0.5, **test_results}])
test_results.to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False)
joblib.dump(selected_model, MODEL_DIR / "selected_final_model.joblib")

test_probability = selected_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)
confusion = pd.DataFrame(confusion_matrix(y_test, test_prediction, labels=[0, 1]),
                         index=["actual_paid", "actual_default"],
                         columns=["predicted_paid", "predicted_default"])
confusion.to_csv(OUTPUT_DIR / "final_test_confusion_matrix.csv")

fpr, tpr, _ = roc_curve(y_test, test_probability)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f"ROC-AUC={test_results.loc[0, 'roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="Held-out test ROC")
axes[0].legend()
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set(title="Held-out test confusion matrix", xlabel="Predicted", ylabel="Actual")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "final_test_performance.png", dpi=160, bbox_inches="tight")
plt.show()
display(test_results, confusion)

In [ ]:
if selected_name == "logistic_regression_baseline":
    print("Selected model is logistic regression; use its coefficients/odds ratios for interpretation.")
else:
    preprocessor = selected_model.named_steps["preprocessor"]
    tree_model = selected_model.named_steps["model"]
    feature_names = preprocessor.get_feature_names_out()
    background_source = X_train.sample(n=min(200, len(X_train)), random_state=RANDOM_STATE)
    explain_source = X_validation.sample(n=min(1_000, len(X_validation)), random_state=RANDOM_STATE)
    background = preprocessor.transform(background_source)
    explain_data = preprocessor.transform(explain_source)
    if hasattr(background, "toarray"):
        background = background.toarray()
    if hasattr(explain_data, "toarray"):
        explain_data = explain_data.toarray()
    explainer = shap.TreeExplainer(tree_model, data=background,
                                   feature_perturbation="interventional",
                                   feature_names=feature_names)
    explanation = explainer(explain_data)
    shap_values = explanation.values
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]
    importance = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names,
                           name="mean_absolute_shap").sort_values(ascending=False)
    importance.to_csv(OUTPUT_DIR / "shap_feature_importance.csv")
    shap.summary_plot(shap_values, explain_data, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figures" / "shap_summary.png", dpi=160, bbox_inches="tight")
    plt.show()
    display(importance.head(20))